# **Setup**

In [1]:
import os, sys

# Repository information
REPO_NAME = "RecSys-Challenge-2025"
REPO_URL  = f"github.com/Lv1g1/{REPO_NAME}.git"

# Detect environment
IS_COLAB = 'content' in os.getcwd()
IS_KAGGLE = 'kaggle' in os.getcwd()
IS_LOCAL = not (IS_COLAB or IS_KAGGLE)

WORKING_DIR = os.getcwd()

if IS_COLAB:
    WORKING_DIR = "/content"

    # Mount Google Drive
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

    # Get GitHub token via input
    def get_token():
        from getpass import getpass
        return getpass("GitHub Token: ")

elif IS_KAGGLE:
    WORKING_DIR = "/kaggle/working"

    # Get GitHub token from Kaggle secrets
    def get_token():
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("Token")

# If local environment assume inside the repo
LOCAL_REPO_PATH = "/home/luigi/RecSys" if IS_LOCAL else os.path.join(WORKING_DIR, REPO_NAME)

# Clone the repository if it doesn't exist
if not os.path.exists(LOCAL_REPO_PATH):
    os.chdir(WORKING_DIR)
    token = get_token()

    !git clone https://{token}@{REPO_URL}
else:
    print("Repo already exists — pulling latest changes")
    os.chdir(LOCAL_REPO_PATH)
    !git pull
    os.chdir(WORKING_DIR)

# Add to Python PATH
if LOCAL_REPO_PATH not in sys.path:
    sys.path.append(LOCAL_REPO_PATH)

if IS_COLAB:
    !pip install optuna

Repo already exists — pulling latest changes
Already up to date.


In [2]:
if IS_COLAB or False:  # Set to True if you want to recompile Cython files
    os.chdir(LOCAL_REPO_PATH)
    !python run_compile_all_cython.py
    os.chdir(WORKING_DIR)

In [3]:
import numpy as np
import optuna

from Challenge.paths import load_cv_folds
from Challenge.hyper_tuning import ModelOptimizer
from Challenge.utils import evaluate_recommender

Running on local — storage at: /home/luigi/RecSys


/home/luigi/RecSys/Challenge/hyper_tuning.py:75: ExperimentalWarning: WilcoxonPruner is experimental (supported from v3.6.0). The interface can change in the future.
  def create_study(self, study_name, direction="maximize", load_if_exists=True, pruner=optuna.pruners.WilcoxonPruner()):
/home/luigi/RecSys/Challenge/hyper_tuning.py:107: ExperimentalWarning: WilcoxonPruner is experimental (supported from v3.6.0). The interface can change in the future.
  def create_and_optimize_study(self, study_name, objective_function, n_trials=50, direction="maximize", load_if_exists=True, pruner=optuna.pruners.WilcoxonPruner()):


# **Load Data**

In [4]:
# Load datasets
folds = load_cv_folds(5)

from Recommenders.MatrixFactorization.Cython.MatrixFactorization_Cython import MatrixFactorization_BPR_Cython

optimizer = ModelOptimizer("MatrixFactorization_BPR")

# **Hyperparameter search**

In [16]:
STUDY_NAME = MatrixFactorization_BPR_Cython.RECOMMENDER_NAME + "_epochs_v3"

In [17]:
URM_train, URM_val = folds[0]

# Fast tuning to find the best regions, no cv
def objective_function(optuna_trial: optuna.trial.Trial) -> float:

    params = {
        "batch_size": 512,
        "sgd_mode": "adagrad",
        "learning_rate": 0.04929677119554038,
        "num_factors": 96,
        "user_reg": 1.005636573502989e-06,
        "positive_reg": 1.005636573502989e-06,
        "negative_reg": 0.00014922048976360032,
        "positive_threshold_BPR": None,
        "epochs": optuna_trial.suggest_int("epochs", 100, 2000, step=100)
    }

    # Train the recommender
    recommender_instance = MatrixFactorization_BPR_Cython(URM_train)
    recommender_instance.fit(**params)
    
    # Evaluate
    score = evaluate_recommender(recommender_instance, at=20, URM_validation=URM_val)
        
    # Log folds performance
    optimizer.log_folds([score], params)

    # Return the mean CV score for the fully completed trial
    return score

In [19]:
optuna_study = optimizer.create_and_optimize_study(
    study_name=STUDY_NAME,
    objective_function=objective_function,
    n_trials=0
)

[I 2025-11-28 19:59:54,501] Using an existing study with name 'MatrixFactorization_BPR_Cython_Recommender_epochs_v3' instead of creating a new one.



Study statistics: 
  Number of finished trials:  12
  Number of pruned trials:  0
  Number of complete trials:  11

Best Value: 0.17914202582353736
Best Params: {'epochs': 1700}


In [20]:
optuna.visualization.plot_optimization_history(optuna_study)

In [22]:
optuna.visualization.plot_parallel_coordinate(optuna_study)

## **2**

In [ ]:
STUDY_NAME = MatrixFactorization_BPR_Cython.RECOMMENDER_NAME + "_epochs_v1"

In [ ]:
def objective_function(optuna_trial: optuna.trial.Trial) -> float:
    params = {
        "batch_size": 512,
        "sgd_mode": "adagrad",
        "learning_rate": 0.04929677119554038,
        "num_factors": 96,
        "user_reg": 1.005636573502989e-06,
        "positive_reg": 1.005636573502989e-06,
        "negative_reg": 0.00014922048976360032,
        "positive_threshold_BPR": None,
        "epochs": optuna_trial.suggest_int("epochs", 100, 1000, step=25)
    }

    validation_scores = []
    for fold_idx, (URM_train, URM_val) in enumerate(folds):
        # Train the recommender
        recommender_instance = MatrixFactorization_BPR_Cython(URM_train)
        recommender_instance.fit(**params)
        
        # Evaluate
        score = evaluate_recommender(recommender_instance, at=10, URM_validation=URM_val)
        validation_scores.append(score)
        
        # Show fold result
        print(f"  Fold {fold_idx+1}/{len(folds)} - Score: {score}")

        # Report intermediate result to Optuna
        optuna_trial.report(score, fold_idx)

        # Ask Optuna to prune if performance is poor
        if optuna_trial.should_prune():
            # Return the average score so far instead of raising TrialPruned,
            # which is a common workaround for WilcoxonPruner.
            return np.mean(validation_scores)
        
    # Log folds performance
    optimizer.log_folds(validation_scores, params)

    # Return the mean CV score for the fully completed trial
    return np.mean(validation_scores)

In [ ]:
optuna_study = optimizer.create_and_optimize_study(
    study_name=STUDY_NAME,
    objective_function=objective_function,
    n_trials=20
)

In [ ]:
optuna.visualization.plot_optimization_history(optuna_study)

In [ ]:
optuna.visualization.plot_param_importances(optuna_study)

In [ ]:
optuna.visualization.plot_parallel_coordinate(optuna_study)

## **CV**

In [ ]:
def objective_function(optuna_trial: optuna.trial.Trial) -> float:
    params = {
        "batch_size": 512,
        "sgd_mode": "adagrad",
        "learning_rate": 0.04929677119554038,
        "num_factors": 96,
        "user_reg": 1.005636573502989e-06,
        "positive_reg": 1.005636573502989e-06,
        "negative_reg": 0.00014922048976360032,
        "positive_threshold_BPR": None,
        "epochs": optuna_trial.suggest_int("epochs", 100, 1000, step=25)
    }

    validation_scores = []
    for fold_idx, (URM_train, URM_val) in enumerate(folds):
        # Train the recommender
        recommender_instance = MatrixFactorization_BPR_Cython(URM_train)
        recommender_instance.fit(**params)
        
        # Evaluate
        score = evaluate_recommender(recommender_instance, at=10, URM_validation=URM_val)
        validation_scores.append(score)
        
        # Show fold result
        print(f"  Fold {fold_idx+1}/{len(folds)} - Score: {score}")

        # Report intermediate result to Optuna
        optuna_trial.report(score, fold_idx)

        # Ask Optuna to prune if performance is poor
        if optuna_trial.should_prune():
            # Return the average score so far instead of raising TrialPruned,
            # which is a common workaround for WilcoxonPruner.
            return np.mean(validation_scores)
        
    # Log folds performance
    optimizer.log_folds(validation_scores, params)

    # Return the mean CV score for the fully completed trial
    return np.mean(validation_scores)

## **Best Model**
- ADD HERE